In this part, we focus on the CausalLM and there is no cross attention and token embedding

In [1]:
import math
import torch
import torch.nn as nn

import warnings
warnings.filterwarnings(action="ignore")

In [27]:
class SimpleDecoder(nn.Module):
    def __init__(self, hidden_dim, nums_head, dropout = 0.1):
        super().__init__()

        self.nums_head = nums_head
        self.head_dim = hidden_dim // nums_head

        self.dropout = dropout

        # 这里按照 transformers 中的 decoder 来写，用 post_norm 的方式实现注意有 残差链接
        # eps 是为了防止溢出；其中 llama 系列的模型一般用的是 RMSnorm 以及 pre-norm（为了稳定性）
        # RMSnorm 没有一个 recenter 的操作，而 layernorm 是让模型重新变成 均值为 0，方差为 1
        # RMS 使用 w平方根均值进行归一化 $\sqrt{\frac{1}{n} \sum_{1}^{n}{a_i^2} }$
        self.layernorm_att = nn.LayerNorm(hidden_dim, eps = 1e-5)

        self.q_proj = nn.Linear(hidden_dim, hidden_dim)
        self.k_proj = nn.Linear(hidden_dim, hidden_dim)
        self.v_proj = nn.Linear(hidden_dim, hidden_dim)
        self.o_proj = nn.Linear(hidden_dim, hidden_dim) # final output for the MHP

        self.drop_att = nn.Dropout(self.dropout)

        # for the fnn preparation
        self.layernorm_ffn = nn.LayerNorm(hidden_dim, eps = 1e-5)
        self.up_proj = nn.Linear(hidden_dim, 4 * hidden_dim)
        self.down_proj = nn.Linear(4 * hidden_dim, hidden_dim)
        self.act_fn = nn.ReLU()

        self.drop_ffn = nn.Dropout(self.dropout)


    def attention_output(self, query, key, value, attention_mask = None):
        # 计算两者相关性
        key = key.transpose(-1, -2)  # (batch, num_head, head_dim, seq)
        att_weight = torch.matmul(query, key) / math.sqrt(self.head_dim)
            
        # attention mask 进行依次调整；变成 causal_attention
        if attention_mask is not None:
            # 变成下三角矩阵
            attention_mask = attention_mask.tril()
            att_weight = att_weight.masked_fill(attention_mask == 0, float('-inf'))
        else:
            # 人工构造一个下三角的 attention mask
            attention_mask = torch.ones_like(att_weight).tril()
            att_weight = att_weight.masked_fill(attention_mask == 0, float('-inf'))

        att_weight = torch.softmax(att_weight, dim = -1)
        att_weight = self.drop_att(att_weight)
        # print("att_weight shape is: ", att_weight)
        # mid_output shape is: (batch, nums_head, seq, head_dim)
        mid_output = torch.matmul(att_weight, value)
        # print("mid_output shape is: ", mid_output)

        mid_output = mid_output.transpose(1, 2).contiguous() # (batch, seq, nums_head, seq)
        batch, seq, _, _ = mid_output.size()

        mid_output = mid_output.view(batch, seq, -1) # (batch, seq, hidden_dim)
        output = self.o_proj(mid_output)
        return output


    def attention_block(self, X, attention_mask = None):
        batch, seq, _ = X.size()
        query = self.q_proj(X).view(batch, seq, self.nums_head, self.head_dim).transpose(1, 2) # (batch, nums_head, seq, head_dim)
        key = self.k_proj(X).view(batch, seq, self.nums_head, self.head_dim).transpose(1, 2) # (batch, nums_head, seq, head_dim)
        value = self.v_proj(X).view(batch, seq, self.nums_head, self.head_dim).transpose(1, 2) # (batch, nums_head, seq, head_dim)

        att_output = self.attention_output(
            query, 
            key, 
            value, 
            attention_mask=attention_mask)
        return self.layernorm_att(X + att_output)


    def ffn_block(self, X):
        up = self.up_proj(X)
        act = self.act_fn(up)
        down = self.down_proj(act)

        # dropout
        down = self.drop_ffn(down)

        # 进行 norm 操作
        return self.layernorm_ffn(X + down)

    def forward(self, X, attention_mask = None):
        # X 一般假设是已经经过 embedding 的输入， (batch, seq, hidden_dim)
        # attention_mask 一般指的是 tokenizer 后返回的 mask 结果，表示哪些样本需要忽略
        # shape 一般是： (batch, nums_head, seq)
        att_output = self.attention_block(X, attention_mask=attention_mask)
        ffn_output = self.ffn_block(att_output)
        return ffn_output


In [28]:
x = torch.rand(3, 4, 64)
net = SimpleDecoder(64, 8)
mask = (
    torch.tensor([[1, 1, 1, 1], [1, 1, 0, 0], [1, 1, 1, 0]])
    .unsqueeze(1)
    .unsqueeze(2)
    .repeat(1, 8, 4, 1)
)

In [29]:
net(x).shape

torch.Size([3, 4, 64])

In [30]:
net(x, mask).shape

torch.Size([3, 4, 64])